In [5]:
!pip install ensemble-boxes

In [4]:
from google.colab import drive
import os
import pandas as pd
import json
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion

In [6]:
#디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [7]:
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
zip_path = '/content/drive/MyDrive/new_dataset.zip'
extract_path = '/content/yolo_dataset'

!unzip -q -o "{zip_path}" -d "{extract_path}"

In [9]:
#경로설정
train_img_path = os.path.join(extract_path, 'train', 'images')
train_label_path = os.path.join(extract_path, 'train', 'labels')
val_img_path = os.path.join(extract_path, 'val', 'images')
val_label_path = os.path.join(extract_path, 'val', 'labels')
test_img_path = os.path.join(extract_path, 'test', 'images')

paths = {
    "Train 이미지": train_img_path,
    "Train 라벨": train_label_path,
    "Val 이미지": val_img_path,
    "Val 라벨": val_label_path,
    "Test 이미지": test_img_path
}

for name, path in paths.items():
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"{name}: {count}개")
    else:
        print(f"{name}: 경로 없음")

# 최종 합계 (이미지 기준)
total_images = len(os.listdir(train_img_path)) + len(os.listdir(val_img_path))
print(f"총 이미지 합계: {total_images}개")

Train 이미지: 295개
Train 라벨: 295개
Val 이미지: 37개
Val 라벨: 37개
Test 이미지: 842개
총 이미지 합계: 332개


In [10]:
def csv_to_wbf_json(csv_path, json_output_path):
    df = pd.read_csv(csv_path)
    json_data = []

    for _, row in df.iterrows():
        # CSV의 x, y, w, h를 x1, y1, x2, y2로 변환
        x1 = float(row['bbox_x'])
        y1 = float(row['bbox_y'])
        x2 = x1 + float(row['bbox_w'])
        y2 = y1 + float(row['bbox_h'])

        json_data.append({
            "image_id": int(row['image_id']),
            "category_id": int(row['category_id']),
            "bbox": [x1, y1, x2, y2],
            "score": float(row['score'])
        })

    with open(json_output_path, 'w') as f:
        json.dump(json_data, f, indent=4)
    print(f"✅ 변환 완료: {json_output_path}")

# 실행 예시
csv_to_wbf_json('/content/drive/MyDrive/yolov11m_submission.csv', 'yolo_predictions.json')
csv_to_wbf_json('/content/drive/MyDrive/submission_rtdetrv4_hflip_tta.csv', 'rtdetr_predictions.json')

✅ 변환 완료: yolo_predictions.json
✅ 변환 완료: rtdetr_predictions.json


In [79]:
def json_based_ensemble():
    SUB_W, SUB_H = 976, 1280
    ID_LIST = [1900, 16548, 19607, 29451, 33009, 21771, 27926, 24850, 29345, 16551, 33208, 2483, 3743, 12778, 13395, 12081, 25438, 19552, 22362, 3351, 3832, 16232, 16262, 16688, 20238, 21325, 22074, 29667, 35206, 36637, 38162, 13900, 18357, 19232, 20014, 31863, 32310, 33880, 41768, 18147, 3483, 20877, 30308, 34597, 22347, 25469, 19861, 28763, 27733, 25367, 31885, 27777, 3544, 4543, 12247, 27993]
    master_id_to_idx = {id_val: i for i, id_val in enumerate(ID_LIST)}

    with open("yolo_predictions.json", 'r') as f: yolo_data = json.load(f)
    with open("rtdetr_predictions.json", 'r') as f: rt_data = json.load(f)

    img_ids = sorted(list(set([p['image_id'] for p in yolo_data] + [p['image_id'] for p in rt_data])))
    final_rows = []

    for img_id in tqdm(img_ids, desc="JSON WBF 진행 중"):
        boxes_list, scores_list, labels_list = [], [], []

        # 각 소스(YOLO, RT-DETR)별로 해당 이미지의 데이터를 정규화하여 준비
        for source_data in [yolo_data, rt_data]:
            b_tmp, s_tmp, l_tmp = [], [], []
            for p in source_data:
                if p['image_id'] == img_id:
                    if p['score'] < 0.01: continue # 낮은 점수 사전 필터링

                    # 1. 좌표 정규화 (가장 중요: 제출 규격 기준)
                    x1, y1, x2, y2 = p['bbox']
                    b_tmp.append([
                        max(0, x1 / SUB_W), max(0, y1 / SUB_H),
                        min(1, x2 / SUB_W), min(1, y2 / SUB_H)
                    ])
                    s_tmp.append(float(p['score']))
                    l_tmp.append(master_id_to_idx[p['category_id']])

            if b_tmp:
                boxes_list.append(b_tmp)
                scores_list.append(s_tmp)
                labels_list.append(l_tmp)

        if not boxes_list: continue

        # WBF 실행
        merged_boxes, merged_scores, merged_labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=[5, 10],
            iou_thr=0.55,
            skip_box_thr=0.05,
            conf_type='max'
        )

        # 결과 복구 및 저장
        for box, score, label in zip(merged_boxes, merged_scores, merged_labels):
            if score < 0.1: continue
            x1, y1, x2, y2 = box
            final_rows.append({
                'image_id': img_id,
                'category_id': ID_LIST[int(label)],
                'bbox_x': round(float(x1 * SUB_W), 2),
                'bbox_y': round(float(y1 * SUB_H), 2),
                'bbox_w': round(float((x2 - x1) * SUB_W), 2),
                'bbox_h': round(float((y2 - y1) * SUB_H), 2),
                'score': round(min(1.0, float(score)), 4) # 1.0 초과 방지
            })

    # CSV 저장
    df = pd.DataFrame(final_rows)
    df.insert(0, 'annotation_id', range(1, len(df) + 1))
    df.to_csv('rtdetr_ensemble_submission.csv', index=False)
    print("✅ JSON 기반 앙상블 CSV 저장 완료!")

json_based_ensemble()

JSON WBF 진행 중: 100%|██████████| 842/842 [00:03<00:00, 216.36it/s]


✅ JSON 기반 앙상블 CSV 저장 완료!
